# Day 55 — RAG Fundamentals
### Chunking strategies, embeddings, ChromaDB vector store, semantic retrieval, full RAG pipeline

## 1. Install Dependencies

In [1]:
import sys
!{sys.executable} -m pip install chromadb sentence-transformers --quiet


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Setup & Imports

In [2]:
import chromadb
import anthropic
from sentence_transformers import SentenceTransformer
import re

client = anthropic.Anthropic()
MODEL = "claude-haiku-4-5"
embedder = SentenceTransformer("all-MiniLM-L6-v2")

print("ChromaDB version:", chromadb.__version__)
print("Embedding model loaded:", embedder)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

c:\DS-AI-75d\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\vijen\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

ChromaDB version: 1.5.9
Embedding model loaded: SentenceTransformer(
  (0): Transformer({'transformer_task': 'feature-extraction', 'modality_config': {'text': {'method': 'forward', 'method_output_name': 'last_hidden_state'}}, 'module_output_name': 'token_embeddings', 'architecture': 'BertModel'})
  (1): Pooling({'embedding_dimension': 384, 'pooling_mode': 'mean', 'include_prompt': True})
  (2): Normalize({})
)


## 3. Our Document Corpus

In [3]:
documents = [
    {
        "id": "doc1",
        "title": "Python Programming",
        "text": """Python is a high-level, interpreted programming language known for its simplicity and readability.
It was created by Guido van Rossum and first released in 1991. Python supports multiple programming
paradigms including procedural, object-oriented, and functional programming. It has a large standard
library and a vibrant ecosystem of third-party packages. Python is widely used in web development,
data science, artificial intelligence, automation, and scientific computing. The language emphasizes
code readability with its use of significant whitespace and clean syntax."""
    },
    {
        "id": "doc2",
        "title": "Machine Learning",
        "text": """Machine learning is a subset of artificial intelligence that enables systems to learn and improve
from experience without being explicitly programmed. It focuses on developing algorithms that can
access data and use it to learn for themselves. The main types are supervised learning, unsupervised
learning, and reinforcement learning. Common algorithms include linear regression, decision trees,
random forests, support vector machines, and neural networks. Machine learning is used in spam
detection, image recognition, recommendation systems, and natural language processing."""
    },
    {
        "id": "doc3",
        "title": "Large Language Models",
        "text": """Large language models (LLMs) are deep learning models trained on massive text datasets to understand
and generate human language. They use transformer architecture with attention mechanisms to process
text. Examples include GPT-4, Claude, and Gemini. LLMs are trained using self-supervised learning
on billions of tokens of text. They can perform tasks like text generation, translation, summarisation,
question answering, and code generation. Fine-tuning adapts pre-trained LLMs for specific tasks.
Prompt engineering is the practice of crafting effective inputs to get desired outputs from LLMs."""
    },
    {
        "id": "doc4",
        "title": "Vector Databases",
        "text": """Vector databases are specialised databases designed to store and query high-dimensional vector
embeddings efficiently. Unlike traditional databases that use exact matching, vector databases use
approximate nearest neighbour search to find semantically similar items. Popular vector databases
include Pinecone, Weaviate, Qdrant, and ChromaDB. They are essential for semantic search, RAG
systems, and recommendation engines. Vectors are indexed using algorithms like HNSW (Hierarchical
Navigable Small World) for fast approximate search. ChromaDB is an open-source, embeddable vector
database commonly used for local development and prototyping."""
    }
]

print(f"Corpus: {len(documents)} documents")
for doc in documents:
    print(f"  {doc['id']}: {doc['title']} ({len(doc['text'].split())} words)")

Corpus: 4 documents
  doc1: Python Programming (78 words)
  doc2: Machine Learning (78 words)
  doc3: Large Language Models (83 words)
  doc4: Vector Databases (82 words)


## 4. Chunking Strategies

In [4]:
def fixed_size_chunk(text, chunk_size=100, overlap=20):
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunks.append(" ".join(words[start:end]))
        start += chunk_size - overlap
    return chunks

def recursive_chunk(text, chunk_size=100, overlap=20):
    separators = ["\n\n", "\n", ". ", " "]
    chunks = []

    def split(text, separators):
        if len(text.split()) <= chunk_size:
            return [text]
        for sep in separators:
            if sep in text:
                parts = text.split(sep)
                result = []
                current = ""
                for part in parts:
                    candidate = current + sep + part if current else part
                    if len(candidate.split()) <= chunk_size:
                        current = candidate
                    else:
                        if current:
                            result.append(current.strip())
                        current = part
                if current:
                    result.append(current.strip())
                return result
        return [text]

    raw_chunks = split(text, separators)
    for i, chunk in enumerate(raw_chunks):
        if i > 0 and overlap > 0:
            prev_words = raw_chunks[i-1].split()[-overlap:]
            chunk = " ".join(prev_words) + " " + chunk
        chunks.append(chunk)
    return chunks

sample = documents[0]["text"]
fixed = fixed_size_chunk(sample)
recursive = recursive_chunk(sample)

print(f"Fixed-size chunking:     {len(fixed)} chunks")
for i, c in enumerate(fixed):
    print(f"  chunk {i+1}: {len(c.split())} words | {c[:60]}...")

print(f"\nRecursive chunking:      {len(recursive)} chunks")
for i, c in enumerate(recursive):
    print(f"  chunk {i+1}: {len(c.split())} words | {c[:60]}...")

Fixed-size chunking:     1 chunks
  chunk 1: 78 words | Python is a high-level, interpreted programming language kno...

Recursive chunking:      1 chunks
  chunk 1: 78 words | Python is a high-level, interpreted programming language kno...


In [5]:
sample = documents[0]["text"]
fixed = fixed_size_chunk(sample, chunk_size=30, overlap=5)
recursive = recursive_chunk(sample, chunk_size=30, overlap=5)

print(f"Fixed-size chunking:     {len(fixed)} chunks")
for i, c in enumerate(fixed):
    print(f"  chunk {i+1}: {len(c.split())} words | {c[:70]}...")

print(f"\nRecursive chunking:      {len(recursive)} chunks")
for i, c in enumerate(recursive):
    print(f"  chunk {i+1}: {len(c.split())} words | {c[:70]}...")

Fixed-size chunking:     4 chunks
  chunk 1: 30 words | Python is a high-level, interpreted programming language known for its...
  chunk 2: 30 words | Python supports multiple programming paradigms including procedural, o...
  chunk 3: 28 words | is widely used in web development, data science, artificial intelligen...
  chunk 4: 3 words | and clean syntax....

Recursive chunking:      3 chunks
  chunk 1: 29 words | Python is a high-level, interpreted programming language known for its...
  chunk 2: 32 words | 1991. Python supports multiple programming paradigms including procedu...
  chunk 3: 27 words | widely used in web development, data science, artificial intelligence,...


## 5. Build the ChromaDB Vector Store

In [6]:
chroma_client = chromadb.Client()
collection = chroma_client.create_collection(name="tech_docs")

all_chunks = []
all_ids = []
all_metadata = []

for doc in documents:
    chunks = fixed_size_chunk(doc["text"], chunk_size=40, overlap=5)
    for i, chunk in enumerate(chunks):
        chunk_id = f"{doc['id']}_chunk{i}"
        all_chunks.append(chunk)
        all_ids.append(chunk_id)
        all_metadata.append({"source": doc["id"], "title": doc["title"], "chunk_index": i})

embeddings = embedder.encode(all_chunks).tolist()

collection.add(
    documents=all_chunks,
    embeddings=embeddings,
    ids=all_ids,
    metadatas=all_metadata
)

print(f"Total chunks stored: {collection.count()}")
print(f"Sample chunk IDs: {all_ids[:4]}")

Total chunks stored: 12
Sample chunk IDs: ['doc1_chunk0', 'doc1_chunk1', 'doc1_chunk2', 'doc2_chunk0']


## 6. Semantic Retrieval

In [7]:
def retrieve(query, n_results=3):
    query_embedding = embedder.encode([query]).tolist()
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=n_results,
        include=["documents", "metadatas", "distances"]
    )
    retrieved = []
    for i in range(len(results["ids"][0])):
        retrieved.append({
            "id": results["ids"][0][i],
            "text": results["documents"][0][i],
            "source": results["metadatas"][0][i]["title"],
            "distance": round(results["distances"][0][i], 4)
        })
    return retrieved

query = "How do vector databases store and search embeddings?"
results = retrieve(query)

print(f"Query: {query}\n")
for r in results:
    print(f"[{r['source']}] (distance: {r['distance']})")
    print(f"  {r['text'][:120]}...")
    print()

Query: How do vector databases store and search embeddings?

[Vector Databases] (distance: 0.4958)
  Vector databases are specialised databases designed to store and query high-dimensional vector embeddings efficiently. U...

[Vector Databases] (distance: 0.6979)
  databases include Pinecone, Weaviate, Qdrant, and ChromaDB. They are essential for semantic search, RAG systems, and rec...

[Vector Databases] (distance: 0.9799)
  an open-source, embeddable vector database commonly used for local development and prototyping....



## 7. Full RAG Pipeline — Retrieval + Generation

In [8]:
def rag_answer(question, n_results=3):
    chunks = retrieve(question, n_results=n_results)
    context = "\n\n".join([f"[{c['source']}]: {c['text']}" for c in chunks])

    prompt = f"""Answer the question using ONLY the context provided below.
If the answer is not in the context, say 'I don't have enough information to answer that.'

Context:
{context}

Question: {question}"""

    response = client.messages.create(
        model=MODEL,
        max_tokens=400,
        messages=[{"role": "user", "content": prompt}]
    )
    return {
        "question": question,
        "answer": response.content[0].text,
        "sources": list(set([c["source"] for c in chunks]))
    }

result = rag_answer("What programming paradigms does Python support?")
print("Q:", result["question"])
print("A:", result["answer"])
print("Sources:", result["sources"])

Q: What programming paradigms does Python support?
A: According to the context, Python supports multiple programming paradigms including:

1. Procedural programming
2. Object-oriented programming
3. Functional programming
Sources: ['Python Programming', 'Machine Learning']


## 8. Test With More Questions

In [9]:
questions = [
    "What is the transformer architecture used for?",
    "What algorithms does ChromaDB use for fast search?",
    "What are the main types of machine learning?",
    "What year was Python first released?"
]

for q in questions:
    result = rag_answer(q)
    print(f"Q: {q}")
    print(f"A: {result['answer']}")
    print(f"Sources: {result['sources']}")
    print("-" * 60)

Q: What is the transformer architecture used for?
A: Based on the context provided, the transformer architecture is used to process text in Large Language Models (LLMs). Specifically, transformers use attention mechanisms to understand and generate human language.
Sources: ['Large Language Models', 'Machine Learning', 'Python Programming']
------------------------------------------------------------
Q: What algorithms does ChromaDB use for fast search?
A: Based on the context provided, I don't have enough information to answer that question. 

The context mentions that ChromaDB is "an open-source, embeddable vector database," but it does not specify what algorithms ChromaDB specifically uses for fast search. While the context does mention that vectors are indexed using algorithms like HNSW (Hierarchical Navigable Small World) for fast approximate search, it doesn't explicitly state that ChromaDB uses this algorithm.
Sources: ['Vector Databases', 'Machine Learning']
--------------------

## 9. Summary — What We Built Today

| Component | What we used | Key insight |
|-----------|-------------|-------------|
| Chunking | Fixed-size (word count) | Simple but creates orphan chunks and cuts mid-sentence |
| Chunking | Recursive (separator-aware) | Respects sentence/paragraph boundaries — cleaner chunks |
| Embedding model | all-MiniLM-L6-v2 (local, free) | Converts text to 384-dim vectors — no API cost |
| Vector store | ChromaDB (in-memory) | Stores embeddings + metadata, queries by cosine similarity |
| Retrieval | Semantic search via distance | Finds relevant chunks even without exact keyword match |
| Generation | Claude Haiku via Anthropic SDK | Answers using ONLY retrieved context — grounded output |

**The full RAG loop:**
1. Chunk documents into smaller pieces
2. Embed each chunk into a vector
3. Store vectors + text in ChromaDB
4. At query time: embed the question, find nearest chunks
5. Pass retrieved chunks as context to the LLM
6. LLM generates an answer grounded in the retrieved context

**Key finding:**
The ChromaDB HNSW question showed correct RAG behaviour — the model said
"I don't have enough information" even though the answer was nearby in the
corpus. Faithfulness to context (Day 54's lesson) showed up naturally here.